# Krylov convergence and the shape of the spectrum

The textbook estimate for the conjugate gradient method is a bound in the
condition number alone. With ρ = (√κ − 1)/(√κ + 1), the error in the
energy norm satisfies

$$\frac{\lVert x - x_k\rVert_A}{\lVert x - x_0\rVert_A} \le 2\rho^k,$$

so a solve to a tolerance ε needs about ½√κ · ln(2/ε) iterations. This is a
worst case over all spectra in [λ_min, λ_max]. CG actually solves a polynomial
approximation problem on the eigenvalues themselves: after k steps the error is
p(A) e₀ for the best polynomial p of degree k with p(0) = 1. Where the
eigenvalues sit inside the interval, and how many distinct ones there are,
decides the iteration count; κ only bounds it.

This notebook:

1. **Holds κ fixed and changes the spectrum.** Five SPD matrices with κ = 10⁴
   and the same random right-hand side take from 2 to several hundred iterations.
2. **Sweeps κ for three spectrum shapes**, against the √κ scaling of the bound.
3. **Shows what finite precision does:** CG on a geometric spectrum takes more
   than n iterations, which exact arithmetic never needs, and in f32 the
   recursively updated residual drifts from the true one.
4. **Compares MINRES on an SPD and an indefinite matrix** with the same
   eigenvalue magnitudes, and shows CG on the indefinite one.

The matrices come from `randspd` and `randsym`, which build a matrix with a
prescribed spectrum. The Krylov solvers are `mtl5.sparse`'s, which take a CSR
matrix in f32 or f64, so the other formats of #166 don't appear here. Every
random matrix and vector has a seed, so the figures are the same on every
build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

import mtl5
import mtl5.sparse as ms

g = mtl5.generators

# Chart styling, shared with the other notebooks.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

N = 200
SEED = 7
b = np.random.default_rng(1).standard_normal(N)


def spectra(kappa, n=N):
    # Five spectra in [1/kappa, 1], so each matrix has condition number kappa.
    k = np.arange(n)
    half = np.linspace(-1, 1, n // 2)
    return {
        "geometric": kappa ** (-k / (n - 1)),
        "arithmetic": 1 - (1 - 1 / kappa) * k / (n - 1),
        "one small": np.r_[np.ones(n - 1), 1 / kappa],
        "one large": np.r_[1.0, np.full(n - 1, 1 / kappa)],
        "two clusters": np.r_[1 - 0.005 * (1 + half), (1 + 0.005 * (1 + half)) / kappa],
    }


def csr(A, dtype=np.float64):
    return sp.csr_matrix(np.asarray(A.to_numpy(), dtype=dtype))


def run(solver, A, rtol=1e-10, maxiter=3000):
    # Every value the solver tests, in order. An iteration count that repeats
    # marks a check of the true residual b - A x after the recurrence's
    # residual claimed convergence (see SolveReport.history).
    calls = []
    x, info, rep = ms.iterative_solve(
        A,
        b,
        solver=solver,
        rtol=rtol,
        maxiter=maxiter,
        full_output=True,
        callback=lambda k, r: calls.append((k, r)),
    )
    return x, info, rep, calls


def cg_bound(kappa, k):
    # The A-norm bound turned into a residual bound: ||r_k|| / ||r_0|| is at most
    # sqrt(kappa) times the A-norm error ratio, so at most 2 sqrt(kappa) rho^k.
    rho = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
    return 2 * np.sqrt(kappa) * rho ** np.asarray(k, dtype=float)

## 1. The same κ, five spectra

Each matrix is `randspd(n, eigenvalues, seed=7)` with n = 200 and its
eigenvalues in [10⁻⁴, 1]:

- **geometric:** spread evenly on a log scale, like `randsvd` mode 3;
- **arithmetic:** spread evenly on a linear scale, like mode 4;
- **one small:** 199 eigenvalues at 1 and one at 10⁻⁴, like mode 2;
- **one large:** one eigenvalue at 1 and 199 at 10⁻⁴, like mode 1;
- **two clusters:** half within 1% of 1, half within 1% of 10⁻⁴.

The strip plot shows where the eigenvalues sit.

In [ ]:
KAPPA = 1e4
spec = spectra(KAPPA)

fig, ax = plt.subplots(figsize=(7.5, 2.6))
for row, (name, lam) in enumerate(spec.items()):
    ax.scatter(lam, np.full(lam.size, row), s=10, color=SERIES[0], alpha=0.5, linewidth=0)
ax.set_xscale("log")
ax.set_yticks(range(len(spec)), list(spec))
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("eigenvalue")
ax.set_title("Five spectra with κ = 10⁴", loc="left", fontsize=11)
fig.tight_layout()
plt.show()

for name, lam in spec.items():
    A = g.randspd(N, list(lam), seed=SEED)
    print(
        f"{name:13s} kappa = {mtl5.condition_number(A):.3g}, "
        f"distinct eigenvalues = {np.unique(lam).size}"
    )

CG solves each one with x₀ = 0 to a relative residual of 10⁻¹⁰. The curves
are the residual the solver tests, which is its recursively updated residual
during the iterations. The dashed line is the κ-only bound
2√κ ρᵏ on the relative residual, which follows from the energy-norm bound
because ‖r‖ ≤ √λ_max ‖e‖_A and ‖e₀‖_A ≤ ‖r₀‖/√λ_min.

In [ ]:
cg_runs = {name: run("cg", csr(g.randspd(N, list(lam), seed=SEED))) for name, lam in spec.items()}

groups = [
    ("Few distinct eigenvalues", ["one small", "one large", "two clusters"]),
    ("Eigenvalues spread over the interval", ["geometric", "arithmetic"]),
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
kmax = max(rep.iterations for _, _, rep, _ in cg_runs.values())
ks = np.arange(kmax + 1)
for ax, (title, names) in zip(axes, groups):
    for color, name in zip(SERIES, names):
        rep = cg_runs[name][2]
        ax.semilogy(
            rep.history / rep.history[0],
            color=color,
            linewidth=2,
            label=f"{name} ({rep.iterations} its)",
        )
    ax.semilogy(
        ks, cg_bound(KAPPA, ks), color=RULE, linestyle="--", linewidth=1.5, label="κ-only bound"
    )
    ax.axhline(1e-10, color=RULE, linestyle=":", linewidth=1)
    ax.set_ylim(1e-12, 1e2)
    ax.set_xlabel("iteration k")
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=8, loc="upper right")
axes[0].set_xlim(0, 40)
axes[0].set_ylabel("‖r_k‖ / ‖b‖")
fig.tight_layout()
plt.show()

rho = (np.sqrt(KAPPA) - 1) / (np.sqrt(KAPPA) + 1)
bound_its = int(np.ceil(np.log(1e-10 / (2 * np.sqrt(KAPPA))) / np.log(rho)))
print(f"the bound reaches 1e-10 after {bound_its} iterations")
for name, (x, info, rep, _) in cg_runs.items():
    print(
        f"{name:13s} info = {info}, iterations = {rep.iterations:4d}, "
        f"true relative residual = {rep.relative_residual:.1e}"
    )

At the same κ, the iteration counts differ by a factor of several hundred.

- **One small and one large** each have two distinct eigenvalues, and CG
  finishes in two steps: in exact arithmetic, CG terminates after at most as
  many steps as A has distinct eigenvalues, since a polynomial of that degree
  can vanish on all of them.
- **Two clusters** take a few more steps: a low-degree polynomial is small on
  two short intervals, and κ barely matters.
- **Arithmetic** is the case the bound is built for: eigenvalues fill the whole
  interval. It still converges well inside the bound: CG converges
  superlinearly, because once it has resolved the extreme eigenvalues, the
  rest behave like a better-conditioned problem (van der Sluis and van der
  Vorst).
- **Geometric** is the slowest. Its residual first *grows*: CG minimizes the
  energy norm of the error, not the residual, so the residual need not decrease.
  It then takes 692 iterations, more than n = 200, which exact arithmetic never
  needs. Section 3 comes back to this.

κ alone predicts none of these counts. It gives the worst case, and only the
geometric spectrum comes near it.

## 2. Iterations against κ

For three of the shapes, the sweep runs κ from 10 to 10⁶ and counts the
iterations CG needs for a relative residual of 10⁻⁸. The dashed line is the
count the bound predicts, which grows like √κ.

In [ ]:
kappas = np.logspace(1, 6, 11)
shapes = ["geometric", "arithmetic", "two clusters"]
its = {name: [] for name in shapes}
for kappa in kappas:
    lam_k = spectra(kappa)
    for name in shapes:
        A = csr(g.randspd(N, list(lam_k[name]), seed=SEED))
        _, info, rep, _ = run("cg", A, rtol=1e-8, maxiter=10000)
        its[name].append(rep.iterations if info == 0 else np.nan)


def bound_iterations(kappa, eps):
    rho = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
    return np.ceil(np.log(eps / (2 * np.sqrt(kappa))) / np.log(rho))


fig, ax = plt.subplots(figsize=(7, 4))
for color, name in zip(SERIES, shapes):
    ax.loglog(kappas, its[name], color=color, linewidth=2, marker="o", markersize=5, label=name)
ax.loglog(
    kappas,
    bound_iterations(kappas, 1e-8),
    color=RULE,
    linestyle="--",
    linewidth=1.5,
    label="κ-only bound",
)
ax.axhline(N, color=RULE, linestyle=":", linewidth=1)
ax.annotate(
    "n = 200", (kappas[0], N), textcoords="offset points", xytext=(2, 4), fontsize=8, color=INK_2
)
ax.set_xlabel("κ")
ax.set_ylabel("CG iterations to ‖r‖/‖b‖ ≤ 10⁻⁸")
ax.set_title("Iterations against κ for three spectrum shapes", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

for name in shapes:
    print(f"{name:13s}", [int(v) if np.isfinite(v) else None for v in its[name]])

The three shapes scale differently with κ:

- **two clusters** grows slowly, from 9 to 17 iterations across five decades
  of κ. Clustering, not κ, governs it, which is why a preconditioner that
  clusters the spectrum works even when it does not reduce κ much.
- **arithmetic** levels off at about 100 iterations. Only its smallest
  eigenvalue, 1/κ, moves with κ. The next one stays near 1/199, so once CG
  has resolved the smallest, the rest has a condition number of about 200
  whatever κ is.
- **geometric** grows almost as fast as the bound, and needs more than n
  iterations from κ = 10³ on.

## 3. Finite precision

In exact arithmetic the residuals CG produces are orthogonal, so there can be
at most n nonzero ones and CG terminates within n steps. In floating point the
orthogonality is lost as the iteration converges toward the extreme
eigenvalues. Finite-precision CG then behaves like exact CG on a larger matrix
whose eigenvalues lie in tiny clusters around those of A, and convergence is
*delayed*, not stopped (Greenbaum 1989; Greenbaum and Strakoš 1992). The geometric
spectrum, with its many well-separated small eigenvalues, is where the delay is
largest.

The second effect is *attainable accuracy*. The recursively updated residual
r_k drifts away from the true residual b − A x_k. The recurrence can keep
decreasing while the true residual stalls near the rounding-error level
u‖A‖‖x‖/‖b‖ of the format. MTL5's CG checks the true residual whenever the
recurrence claims convergence, and restarts from the current x if the true
residual has not converged.

Below, the geometric matrix with κ = 10⁴ is solved in f64 and in f32 with a
tolerance of 10⁻⁶, which f32 cannot reach. The solid line is the tested
residual, the markers are the true-residual checks, and the dotted lines are
the rounding-error level u‖A‖‖x̂‖/‖b‖ of each format, computed from the
returned solution x̂.

In [ ]:
A64 = g.randspd(N, list(spec["geometric"]), seed=SEED)
A_np = A64.to_numpy()
norm_A = np.linalg.norm(A_np, 2)

fig, ax = plt.subplots(figsize=(8, 4))
for color, (label, dtype, u) in zip(
    SERIES, [("f64", np.float64, 2.0**-53), ("f32", np.float32, 2.0**-24)]
):
    x, info, rep, calls = run(
        "cg", csr(A64, dtype), rtol=1e-6 if dtype == np.float32 else 1e-10, maxiter=900
    )
    ax.semilogy(
        rep.history / rep.history[0], color=color, linewidth=2, label=f"{label}: tested residual"
    )
    # The true residual is the second value tested at a repeated iteration count;
    # a restart then tests it once more, so keep one value per count.
    checks = sorted(
        {k: r for (k, r), (k_prev, _) in zip(calls[1:], calls[:-1]) if k == k_prev}.items()
    )
    if checks:
        ck, cr = np.array(checks).T
        ax.scatter(
            ck,
            cr / rep.history[0],
            s=40,
            color=color,
            edgecolor=SURFACE,
            linewidth=2,
            zorder=4,
            label=f"{label}: true residual at a check",
        )
    level = u * norm_A * np.linalg.norm(np.asarray(x, dtype=np.float64)) / np.linalg.norm(b)
    ax.axhline(level, color=color, linestyle=":", linewidth=1.2)
    print(
        f"{label}: info = {info}, iterations = {rep.iterations}, "
        f"true relative residual = {rep.relative_residual:.1e}, "
        f"u‖A‖‖x‖/‖b‖ = {level:.1e}"
    )
    for k, r in checks:
        print(f"   true-residual check at iteration {k}: {r / rep.history[0]:.1e}")
ax.axvline(N, color=RULE, linestyle="--", linewidth=1)
ax.annotate("n = 200", (N, 1e1), textcoords="offset points", xytext=(4, 0), fontsize=8, color=INK_2)
ax.set_ylim(1e-13, 1e2)
ax.set_xlabel("iteration k")
ax.set_ylabel("‖r_k‖ / ‖b‖")
ax.set_title("CG on the geometric spectrum, κ = 10⁴, in f64 and f32", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

Both precisions pass n = 200 iterations well before converging, the delay that
lost orthogonality causes.

In f32 the recurrence reaches the tolerance 10⁻⁶ at iteration 600, but the
true residual there is 5.8·10⁻⁴, near f32's rounding-error level of
1.9·10⁻⁴. CG restarts from the current x, and the recurrence again reaches the
tolerance while the true residual stays near that level, until the iteration
limit: the format, not the iteration count, limits the accuracy. f64's level is about
10⁻¹³, below its tolerance of 10⁻¹⁰, so its single check passes.

## 4. Indefinite systems: MINRES

CG needs A to be positive definite. For a symmetric indefinite A, MINRES
minimizes the residual over the same Krylov space, so its residual never
increases. Its convergence depends on the eigenvalues too, but a polynomial
with p(0) = 1 must now be small on both sides of 0, which is harder.

The comparison below uses two matrices with the same eigenvalue *magnitudes*,
geometrically spaced in [10⁻², 1]:

- **SPD:** all 200 eigenvalues positive, from `randspd`;
- **indefinite:** 100 in [−1, −10⁻²] and 100 in [10⁻², 1], from `randsym`.

Both have κ = 100. The dashed lines are the standard bounds: 2ρᵏ for the SPD
case, and 2((κ − 1)/(κ + 1))^⌊k/2⌋ for two intervals placed symmetrically
about 0, which grows like κ rather than √κ (Greenbaum, *Iterative Methods for
Solving Linear Systems*, ch. 3). CG on the indefinite matrix is shown too.

In [ ]:
mag = np.geomspace(1e-2, 1, N)
indef = np.r_[-mag[::2], mag[1::2]]
A_spd = g.randspd(N, list(mag), seed=SEED)
A_ind = g.randsym(N, list(indef), seed=SEED)
print("SPD inertia:", mtl5.inertia(A_spd), " indefinite inertia:", mtl5.inertia(A_ind))
print(
    f"kappa: SPD {mtl5.condition_number(A_spd):.3g}, indefinite {mtl5.condition_number(A_ind):.3g}"
)

kappa2 = 100.0
cases = [
    ("MINRES, SPD", "minres", A_spd, SERIES[0]),
    ("MINRES, indefinite", "minres", A_ind, SERIES[1]),
    ("CG, indefinite", "cg", A_ind, SERIES[2]),
]
fig, ax = plt.subplots(figsize=(8, 4))
results = {}
for label, solver, A, color in cases:
    x, info, rep, _ = run(solver, csr(A))
    results[label] = (info, rep)
    ax.semilogy(
        rep.history / rep.history[0],
        color=color,
        linewidth=1.5 if solver == "cg" else 2,
        alpha=0.8 if solver == "cg" else 1,
        label=f"{label} ({rep.iterations} its)",
    )
k = np.arange(max(rep.iterations for _, rep in results.values()) + 1)
rho2 = (np.sqrt(kappa2) - 1) / (np.sqrt(kappa2) + 1)
ax.semilogy(k, 2 * rho2**k, color=SERIES[0], linestyle="--", linewidth=1, label="SPD bound")
ax.semilogy(
    k,
    2 * ((kappa2 - 1) / (kappa2 + 1)) ** (k // 2),
    color=SERIES[1],
    linestyle="--",
    linewidth=1,
    label="two-interval bound",
)
ax.axhline(1e-10, color=RULE, linestyle=":", linewidth=1)
ax.set_ylim(1e-12, 1e3)
ax.set_xlabel("iteration k")
ax.set_ylabel("‖r_k‖ / ‖b‖")
ax.set_title("Same |λ|, κ = 100: definite against indefinite", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.tight_layout()
plt.show()

for label, (info, rep) in results.items():
    h = rep.history
    rises = int(np.sum(np.diff(h[np.isfinite(h)]) > 0))
    print(
        f"{label:19s} info = {info}, iterations = {rep.iterations:4d}, "
        f"true relative residual = {rep.relative_residual:.1e}, "
        f"steps where the tested residual rose = {rises}"
    )

The same eigenvalue magnitudes cost about eight times as many MINRES iterations
once half of them are negative. The SPD case converges like √κ = 10, the
indefinite one like κ = 100, as the two bounds predict; both runs finish inside
their bound.

MINRES's residual decreases monotonically in both cases, as its minimization
guarantees. CG on the indefinite matrix happens to converge here, but its
residual rises at many steps, and nothing guarantees it: on an indefinite A,
pᵀAp can be zero for p ≠ 0, which breaks CG down, and MTL5's CG then returns
info = 2. The CG iteration count is not a meaningful comparison with MINRES's
for that reason.

## What to take away

- **κ bounds CG, it doesn't predict it.** On the five spectra of section 1,
  with the same κ, CG took from 2 to 692 iterations. The number of distinct
  eigenvalues, and how they cluster, decides the count.
- **The κ-only bound is a worst case.** Of the tested shapes, only the
  geometric spectrum grows with κ the way the bound does, and only it came
  near the bound.
- **Finite precision delays CG and limits its accuracy.** The geometric
  spectrum needed more than n iterations in f64 and f32. In f32 the
  recursively updated residual reached the tolerance while the true residual
  stayed at f32's rounding-error level.
- **Indefiniteness costs iterations.** With the same eigenvalue magnitudes,
  MINRES on the indefinite matrix took about eight times as many iterations as
  on the SPD one, the κ-against-√κ difference of their bounds. Use MINRES, not
  CG, for symmetric indefinite systems.

### Where to go next

- `examples/krylov_comparison.ipynb`: nine solvers on a nonsymmetric system,
  and what IC(0), ILU(0) and SSOR preconditioning do to the iteration counts on
  the Poisson matrices.
- `spectra_inertia.ipynb`: building symmetric matrices with a chosen spectrum
  and inertia.
- The parametric indefinite generator `randsym(n, kappa, mode, num_negative)`
  (stillwater-sc/mtl5#961) will let section 4 sweep the inertia directly once
  the MTL5 version it ships in is bound.